# NB03: Multi-seed evaluation with dynamic feature-set winner selection

Tests three feature engineering methods (raw, alr, pwlr) across four tree
ensembles (RF, ERT, XGB, GB) for both `opx_liq` and `opx_only` tracks. Tuned
once on seed 42 with `HalvingRandomSearchCV` + `StratifiedGroupKFold`, then
frozen hyperparameters are evaluated on 9 additional split seeds (43-51) so
downstream selection can account for across-split variance.

The old factorial design included `raw_aug`, `alr_aug`, `pwlr_aug` variants.
The n_aug sensitivity appendix at the end of this notebook documents the
decision to drop them: see section "Appendix: N_AUG sensitivity".

Outputs:
- `results/nb03_multi_seed_results.csv`
- `results/nb03_winning_configurations.json`
- `results/nb03_canonical_test_predictions.npz`
- `figures/fig05_model_comparison.png` (downstream, from NB06)


## Purpose / Inputs / Outputs / Canonical decisions

**Purpose:** Multi-seed baseline-model evaluation. Tests three feature engineering methods (raw, alr, pwlr) across four tree ensembles (RF, ERT, XGB, GB) for opx-only and opx-liq tracks under a 10-fold StratifiedGroupKFold CV, repeated across 5 split seeds.

**Inputs:** `data/processed/opx_clean_opx_liq.parquet`, `data/processed/opx_clean_core.csv`, split-index arrays.

**Outputs:** `results/nb03_multi_seed_results.csv`, `results/nb03_multi_seed_summary.csv`, `results/nb03_winning_configurations.json` (read by every downstream notebook via `load_winning_config`), `models/model_<family>_<target>_<track>_<feat>.joblib`.

**Canonical decisions:** `nb03_winning_configurations.json` sets the **global** feature set (`WIN_FEAT`) that all downstream work consumes. Do not hard-code feature names elsewhere - route through `WIN_FEAT` and `canonical_model_filename`.


In [1]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
        OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
    N_SPLIT_REPS, SPLIT_SEEDS, FEATURE_METHODS,
    MODEL_FAMILIES, TIEBREAKER_RULE,
)
import warnings
warnings.filterwarnings('ignore')

import json
import logging
import shutil
import time
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import (
    GroupShuffleSplit, StratifiedGroupKFold, HalvingRandomSearchCV,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# Local vs Colab pathing
if os.path.exists('/content'):
    TEMP_DIR = Path('/content')
else:
    TEMP_DIR = Path.cwd()

FIGS = FIGURES  # alias

In [2]:
# Load cleaned data with chemical cluster labels (written by NB02).
df_all = pd.read_parquet(DATA_PROC / 'opx_clean_core_with_clusters.parquet')
df_all = df_all.reset_index(drop=True)
print(f'Loaded: {len(df_all)} experiments, {df_all["Citation"].nunique()} studies')

df_opx = df_all.copy()
df_liq = df_all[df_all['opx_liq_pair']].reset_index(drop=True).copy()

print(f'Opx-only: {len(df_opx)} rows / {df_opx["Citation"].nunique()} studies')
print(f'Opx-liq:  {len(df_liq)} rows / {df_liq["Citation"].nunique()} studies')


Loaded: 1035 experiments, 123 studies
Opx-only: 1035 rows / 123 studies
Opx-liq:  600 rows / 93 studies


In [3]:
df_liq.to_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
df_opx.to_parquet(DATA_PROC / 'opx_clean_opx_only.parquet')
print('Track parquet files written.')

Track parquet files written.


In [4]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [5]:
# Model definitions and global config. N_SPLIT_REPS, SPLIT_SEEDS,
# FEATURE_METHODS come from config.py. v7 Part G raises N_SPLIT_REPS
# from 10 to 20.
TUNE_SEED = 42
N_AUG = 1  # augmentation disabled per sensitivity test results

HALVING_CV = 3
HALVING_FACTOR = 3
SEARCH_NJOBS = -1     # use all cores for local run
ESTIMATOR_NJOBS = 1   # prevent loky oversubscription

BASE_MODELS = {
    'RF':  lambda: RandomForestRegressor(random_state=SEED_MODEL, n_jobs=ESTIMATOR_NJOBS),
    'ERT': lambda: ExtraTreesRegressor(random_state=SEED_MODEL, n_jobs=ESTIMATOR_NJOBS),
    'XGB': lambda: XGBRegressor(random_state=SEED_MODEL, n_jobs=ESTIMATOR_NJOBS,
                                 verbosity=0, tree_method='hist'),
    'GB':  lambda: HistGradientBoostingRegressor(random_state=SEED_MODEL,
                                                  early_stopping=True,
                                                  validation_fraction=0.15,
                                                  n_iter_no_change=20),
}

PARAM_GRIDS = {
    'RF': {
        'n_estimators': [200, 500, 800],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [0.33, 0.5, 0.66, 'sqrt'],
    },
    'ERT': {
        'n_estimators': [200, 500, 800],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': [0.33, 0.5, 0.66, 'sqrt'],
    },
    'XGB': {
        'n_estimators': [200, 500, 800],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'reg_alpha': [0, 0.1, 1, 10],
        'reg_lambda': [1, 5, 10],
    },
    'GB': {
        'max_iter': [200, 500, 800],
        'max_depth': [3, 5, 7, None],
        'learning_rate': [0.01, 0.05, 0.1],
        'min_samples_leaf': [10, 20, 40],
        'l2_regularization': [0.0, 0.1, 1.0],
        'max_leaf_nodes': [15, 31, 63],
    },
}


def tune_with_halving(X_tr, y_tr, X_te, y_te, model_name, target_name,
                      seed, groups_tr):
    """Seed 42 only. HalvingRandomSearchCV, grouped stratified inner CV."""
    base = BASE_MODELS[model_name]()
    grid = PARAM_GRIDS[model_name]
    y_bins = pd.qcut(y_tr, q=5, labels=False, duplicates='drop')
    sgkf = StratifiedGroupKFold(n_splits=HALVING_CV, shuffle=True,
                                random_state=seed)
    cv_iter = list(sgkf.split(X_tr, y_bins, groups=groups_tr))
    search = HalvingRandomSearchCV(
        base, grid, factor=HALVING_FACTOR, resource='n_samples',
        cv=cv_iter, scoring='neg_mean_squared_error',
        random_state=seed, n_jobs=SEARCH_NJOBS, refit=True,
    )
    search.fit(X_tr, y_tr)
    best = search.best_estimator_
    pred_tr = predict_median(best, X_tr)
    pred_te = predict_median(best, X_te)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
    return {
        'model': best,
        'best_params': search.best_params_,
        'model_name': model_name,
        'target': target_name,
        'rmse_train': rmse_tr,
        'rmse_test': rmse_te,
        'mae_test': mean_absolute_error(y_te, pred_te),
        'r2_test': r2_score(y_te, pred_te),
        'overfit_ratio': rmse_tr / max(rmse_te, 1e-9),
    }


def eval_frozen(X_tr, y_tr, X_te, y_te, model_name, target_name,
                frozen_params):
    """Seeds 43-51. Clone, set frozen params, single fit."""
    est = clone(BASE_MODELS[model_name]())
    est.set_params(**frozen_params)
    est.fit(X_tr, y_tr)
    pred_tr = predict_median(est, X_tr)
    pred_te = predict_median(est, X_te)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
    return {
        'model': est,
        'best_params': frozen_params,
        'model_name': model_name,
        'target': target_name,
        'rmse_train': rmse_tr,
        'rmse_test': rmse_te,
        'mae_test': mean_absolute_error(y_te, pred_te),
        'r2_test': r2_score(y_te, pred_te),
        'overfit_ratio': rmse_tr / max(rmse_te, 1e-9),
    }

## Phase 3.3a: N_AUG sensitivity (pre-training exploration)

Tests `n_aug in {1, 3, 5, 10, 15}` across all three representations, both targets, and both tracks with default RF and XGB hyperparameters, then runs a Wilcoxon signed-rank test (1 vs N) per representation. Augmentation either hurt or failed to significantly help in every cell, so the 3-method design uses N_AUG=1 (identity augmentation).

Writes:
- `results/nb03_n_aug_sensitivity.csv`
- `figures/fig_nb03_n_aug_sensitivity.png`
- `figures/fig_nb03_n_aug_overfit.png`

In [ ]:
# Phase 3.3b: N_AUG sensitivity test
import time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

N_AUG_TEST = [1, 3, 5, 10, 15]
AUG_TEST_MODELS = ['RF', 'XGB']
AUG_TEST_REPRS = ['raw', 'alr', 'pwlr']
AUG_TEST_TARGETS = ['T_C', 'P_kbar']

# v7 hotfix: sensitivity test subsets SPLIT_SEEDS to first 5 seeds.
# Main Phase 3.4 training below still uses all 20 seeds.
_SENS_SEEDS = SPLIT_SEEDS[:5]
n_aug_results = []
total_iters = (len(AUG_TEST_MODELS) * len(N_AUG_TEST) * len(AUG_TEST_REPRS)
               * len(_SENS_SEEDS) * len(AUG_TEST_TARGETS) * 2)
pbar = tqdm(total=total_iters, desc='N_AUG sensitivity')

for track_name, df_track, use_liq in [('opx_liq', df_liq, True),
                                       ('opx_only', df_opx, False)]:
    for seed in _SENS_SEEDS:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        tr_pos, te_pos = next(gss.split(df_track, groups=df_track['Citation'].values))
        df_train = df_track.iloc[tr_pos].copy()
        df_test = df_track.iloc[te_pos].copy()

        for repr_name in AUG_TEST_REPRS:
            X_te, _ = build_feature_matrix(df_test, repr_name, use_liq)

            for n_aug in N_AUG_TEST:
                t0 = time.time()
                if n_aug > 1:
                    df_tr_aug = augment_dataframe(df_train, n_aug=n_aug, seed=seed)
                else:
                    df_tr_aug = df_train.copy()

                X_tr, _ = build_feature_matrix(df_tr_aug, repr_name, use_liq)

                for target in AUG_TEST_TARGETS:
                    y_tr = df_tr_aug[target].values
                    y_te = df_test[target].values

                    for model_name in AUG_TEST_MODELS:
                        est = BASE_MODELS[model_name]()
                        est.fit(X_tr, y_tr)
                        pred_te = predict_median(est, X_te)
                        pred_tr = predict_median(est, X_tr)
                        rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
                        rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))

                        n_aug_results.append({
                            'track': track_name,
                            'model': model_name,
                            'repr': repr_name,
                            'target': target,
                            'n_aug': n_aug,
                            'seed': seed,
                            'rmse_test': rmse_te,
                            'rmse_train': rmse_tr,
                            'r2_test': r2_score(y_te, pred_te),
                            'overfit_ratio': rmse_tr / max(rmse_te, 1e-9),
                        })
                        pbar.update(1)
                        pbar.set_postfix(trk=track_name, m=model_name, r=repr_name,
                                         n=n_aug, s=seed, t=target)

pbar.close()
df_naug = pd.DataFrame(n_aug_results)
df_naug.to_csv(RESULTS / 'nb03_n_aug_sensitivity.csv', index=False)

# ----- Summary table -----
print('=' * 70)
print('N_AUG SENSITIVITY RESULTS (mean +/- std test RMSE)')
print('=' * 70)

summary = (df_naug.groupby(['model', 'track', 'repr', 'target', 'n_aug'])
           .agg(rmse_mean=('rmse_test', 'mean'),
                rmse_std=('rmse_test', 'std'),
                overfit_mean=('overfit_ratio', 'mean'))
           .reset_index())

for (model, track, target), grp in summary.groupby(['model', 'track', 'target']):
    print(f'\n--- {model} | {track} / {target} ---')
    pivot = grp.pivot(index='n_aug', columns='repr', values='rmse_mean')
    pivot_std = grp.pivot(index='n_aug', columns='repr', values='rmse_std')
    for col in pivot.columns:
        pivot[col] = [f'{m:.1f} +/- {s:.1f}'
                      for m, s in zip(pivot[col], pivot_std[col])]
    print(pivot.to_string())

# ----- Wilcoxon signed-rank: n_aug=1 vs each n_aug -----
print('\n' + '=' * 70)
print('WILCOXON SIGNED-RANK TESTS (n_aug=1 vs n_aug=N)')
print('=' * 70)

for (model, track, repr_name, target), grp in df_naug.groupby(['model', 'track', 'repr', 'target']):
    baseline = grp[grp['n_aug'] == 1].sort_values('seed')['rmse_test'].values
    best_rmse, best_n = baseline.mean(), 1
    for n in [3, 5, 10, 15]:
        cand = grp[grp['n_aug'] == n].sort_values('seed')['rmse_test'].values
        if len(cand) == len(baseline):
            delta = baseline - cand
            try:
                stat, p = stats.wilcoxon(delta, alternative='two-sided')
            except ValueError:
                stat, p = np.nan, np.nan
            mean_d = delta.mean()
            direction = 'HELPS' if mean_d > 0 else 'HURTS'
            sig = '*' if p < (0.05 / 4) else ''
            print(f'{model:4s} {track:10s} {repr_name:5s} {target:7s} '
                  f'1 vs {n:2d}: delta={mean_d:+.2f}  p={p:.4f} {sig} {direction}')
            if cand.mean() < best_rmse:
                best_rmse, best_n = cand.mean(), n
    print(f'  -> Best n_aug for {model}/{repr_name}/{target}: {best_n}')

In [ ]:
# ----- Error bar plot: RMSE vs n_aug -----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
panels = [('opx_liq', 'T_C'), ('opx_liq', 'P_kbar'),
          ('opx_only', 'T_C'), ('opx_only', 'P_kbar')]
colors = {'raw': '#1f77b4', 'alr': '#ff7f0e', 'pwlr': '#2ca02c'}
markers = {'RF': 'o', 'XGB': 's'}

for ax, (track, target) in zip(axes.flat, panels):
    for repr_name, color in colors.items():
        for model_name, marker in markers.items():
            sub = df_naug[(df_naug['track'] == track) &
                          (df_naug['target'] == target) &
                          (df_naug['repr'] == repr_name) &
                          (df_naug['model'] == model_name)]
            m = sub.groupby('n_aug')['rmse_test'].mean()
            s = sub.groupby('n_aug')['rmse_test'].std()
            ax.errorbar(m.index, m.values, yerr=s.values, marker=marker,
                        label=f'{repr_name} ({model_name})', color=color,
                        capsize=4, linewidth=2,
                        linestyle='-' if model_name == 'RF' else '--')
    ax.set_xlabel('n_aug')
    ax.set_ylabel('Test RMSE')
    ax.set_title(f'{track} / {target}')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('N_AUG Sensitivity (RF solid, XGB dashed)', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig_nb03_n_aug_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

# ----- Overfit ratio plot -----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (track, target) in zip(axes.flat, panels):
    for repr_name, color in colors.items():
        for model_name, marker in markers.items():
            sub = df_naug[(df_naug['track'] == track) &
                          (df_naug['target'] == target) &
                          (df_naug['repr'] == repr_name) &
                          (df_naug['model'] == model_name)]
            m = sub.groupby('n_aug')['overfit_ratio'].mean()
            s = sub.groupby('n_aug')['overfit_ratio'].std()
            ax.errorbar(m.index, m.values, yerr=s.values, marker=marker,
                        label=f'{repr_name} ({model_name})', color=color,
                        capsize=4, linewidth=2,
                        linestyle='-' if model_name == 'RF' else '--')
    ax.axhline(y=1.0, color='gray', alpha=0.5, linestyle=':')
    ax.set_xlabel('n_aug')
    ax.set_ylabel('Overfit Ratio (train RMSE / test RMSE)')
    ax.set_title(f'{track} / {target}')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('Overfitting vs N_AUG (RF solid, XGB dashed)', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'fig_nb03_n_aug_overfit.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n>>> SET N_AUG in the next cell based on these results <<<')

## Phase 3.3b: Hyperparameter search (pre-training)

HalvingRandomSearchCV on seed 42 for every (track, feature_set, model, target) combo. The resulting best_params are frozen and reused across all 10 split seeds in the final training pass below. Separating search from training makes the hyperparameter decision explicit and auditable in `results/nb03_hyperparameter_search.csv`.

In [ ]:
# Phase 3.3b: Hyperparameter search (HalvingRandomSearchCV on seed 42 only)
# Fits HalvingRandomSearchCV for every (track, feature_set, model, target)
# combination using the tune seed. Persists best_params to frozen_params_store
# and the refit best estimators to seed42_models. The final training pass below
# fits all 10 seeds (including seed 42) with these frozen parameters, so the
# hyperparameter decision is explicit and auditable before any multi-seed work.

# Fresh checkpoint state.
PARTIAL_LOCAL = TEMP_DIR / 'nb03c_partial.csv'
FROZEN_LOCAL = TEMP_DIR / 'nb03c_frozen_params.json'
for p in [PARTIAL_LOCAL, FROZEN_LOCAL]:
    if p.exists():
        p.unlink()

LOG_PATH_SEARCH = TEMP_DIR / 'nb03_search.log'
logger_search = logging.getLogger('nb03_search')
logger_search.setLevel(logging.INFO)
logger_search.handlers.clear()
_fh = logging.FileHandler(LOG_PATH_SEARCH, mode='w')
_fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
logger_search.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter('%(asctime)s | %(message)s', datefmt='%H:%M:%S'))
logger_search.addHandler(_sh)
logger_search.info('Phase 3.3b: HalvingRandomSearchCV | tune_seed=%d', TUNE_SEED)

frozen_params_store = {}
seed42_models = {}
search_records = []

total_combos = 2 * len(FEATURE_METHODS) * 4 * 2
pbar = tqdm(total=total_combos, desc='Hyperparam search (seed 42)', smoothing=0.05)
t0_search = time.time()

for track_name, df_track, use_liq in [('opx_liq', df_liq, True),
                                       ('opx_only', df_opx, False)]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=TUNE_SEED)
    tr_pos, te_pos = next(gss.split(df_track,
                                     groups=df_track['Citation'].values))
    df_train = df_track.iloc[tr_pos].copy()
    df_test  = df_track.iloc[te_pos].copy()

    for feat_name in FEATURE_METHODS:
        X_tr, _ = build_feature_matrix(df_train, feat_name, use_liq)
        X_te, _ = build_feature_matrix(df_test,  feat_name, use_liq)
        groups_tr = df_train['Citation'].values

        for model_name in ['RF', 'ERT', 'XGB', 'GB']:
            for target_name in ['T_C', 'P_kbar']:
                y_tr = df_train[target_name].values
                y_te = df_test[target_name].values
                pbar.set_postfix(trk=track_name, f=feat_name,
                                 m=model_name, t=target_name)
                t0 = time.time()
                try:
                    result = tune_with_halving(
                        X_tr, y_tr, X_te, y_te,
                        model_name, target_name, TUNE_SEED, groups_tr,
                    )
                    frozen_key = (track_name, feat_name, model_name,
                                  target_name)
                    frozen_params_store[frozen_key] = result['best_params']
                    seed42_models[(track_name, target_name,
                                   model_name, feat_name)] = result.pop('model')
                    search_records.append({
                        'track':               track_name,
                        'feature_set':         feat_name,
                        'model_name':          model_name,
                        'target':              target_name,
                        'rmse_test_seed42':    result['rmse_test'],
                        'mae_test_seed42':     result['mae_test'],
                        'r2_test_seed42':      result['r2_test'],
                        'overfit_ratio_seed42': result['overfit_ratio'],
                        'best_params':         str(result['best_params']),
                        'elapsed_s':           round(time.time() - t0, 1),
                    })
                    logger_search.info(
                        '[SEARCH] %s %s %s %s rmse=%.2f r2=%.3f (%.0fs)',
                        track_name, feat_name, model_name, target_name,
                        result['rmse_test'], result['r2_test'],
                        time.time() - t0,
                    )
                except Exception as e:
                    logger_search.error(
                        'FAIL %s/%s/%s/%s: %s: %s',
                        track_name, feat_name, model_name, target_name,
                        type(e).__name__, e,
                    )
                finally:
                    pbar.update(1)

pbar.close()

with open(FROZEN_LOCAL, 'w') as f:
    json.dump({'||'.join(k): {kk: (vv.item() if isinstance(vv, np.generic) else vv)
                              for kk, vv in v.items()}
               for k, v in frozen_params_store.items()}, f, default=str)

search_df = pd.DataFrame(search_records)
search_df.to_csv(RESULTS / 'nb03_hyperparameter_search.csv', index=False)

elapsed_h = (time.time() - t0_search) / 3600
logger_search.info('Search complete: %d combos in %.2fh',
                   len(frozen_params_store), elapsed_h)
print(f'\nSearch complete: {len(frozen_params_store)} combos in {elapsed_h:.2f}h')
print(f'Frozen params persisted to {FROZEN_LOCAL.name}')
print('\nSeed-42 search summary (top of table):')
print(search_df.sort_values(['track', 'target', 'rmse_test_seed42'])
                 .round(3).head(24).to_string(index=False))

## Phase 3.4: Final training (frozen params, 10 seeds)

Expected: 480 fits (2 tracks x 10 seeds x 3 features x 4 models x 2 targets). No retuning. Per-seed RMSE/MAE/R2 written to `results/nb03_multi_seed_results.csv`.

In [ ]:
# Phase 3.3b: canonical split write (v7 Part D).
# Seed-42 train/test indices are persisted to disk BEFORE the 20-seed
# training loop begins. Every downstream phase and notebook loads from
# these NPY files rather than recomputing GroupShuffleSplit.
_gss42_liq = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
_tr_l, _te_l = next(_gss42_liq.split(df_liq, groups=df_liq['Citation'].values))
np.save(DATA_SPLITS / 'train_indices_opx_liq.npy', df_liq.index.values[_tr_l])
np.save(DATA_SPLITS / 'test_indices_opx_liq.npy',  df_liq.index.values[_te_l])
_gss42_opx = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
_tr_o, _te_o = next(_gss42_opx.split(df_opx, groups=df_opx['Citation'].values))
np.save(DATA_SPLITS / 'train_indices_opx.npy', df_opx.index.values[_tr_o])
np.save(DATA_SPLITS / 'test_indices_opx.npy',  df_opx.index.values[_te_o])
print(f'Canonical seed-42 splits written: '
      f'{len(_tr_l)}/{len(_te_l)} liq, {len(_tr_o)}/{len(_te_o)} opx-only')

# Phase 3.4: Final training (frozen params, all 10 seeds)
# Uses frozen_params_store built by the hyperparameter search above. Each
# (track, feature, model, target) combo is fit 10 times with the same frozen
# params on different 80/20 train/test splits (seeds 42-51). No retuning.

LOG_PATH = TEMP_DIR / 'nb03c_training.log'
logger = logging.getLogger('nb03c')
logger.setLevel(logging.INFO)
logger.handlers.clear()
_fh = logging.FileHandler(LOG_PATH, mode='w')
_fh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'))
logger.addHandler(_fh)
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter('%(asctime)s | %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(_sh)
logger.info('=' * 70)
logger.info('Phase 3.4: final training (frozen) | N_AUG=%d | features=%s',
            N_AUG, FEATURE_METHODS)

multi_seed_results = []

total = 2 * N_SPLIT_REPS * len(FEATURE_METHODS) * 4 * 2
pbar = tqdm(total=total, desc='Training (frozen)', smoothing=0.05)
t0_global = time.time()

for track_name, df_track, use_liq in [('opx_liq', df_liq, True),
                                       ('opx_only', df_opx, False)]:
    for seed in SPLIT_SEEDS:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        tr_pos, te_pos = next(gss.split(df_track,
                                         groups=df_track['Citation'].values))
        df_train = df_track.iloc[tr_pos].copy()
        df_test  = df_track.iloc[te_pos].copy()
        assert set(df_train['Citation']).isdisjoint(set(df_test['Citation']))

        for feat_name in FEATURE_METHODS:
            # N_AUG=1 -> augment_dataframe is an identity copy (see 3R.3a).
            df_tr_use = df_train
            X_tr, _ = build_feature_matrix(df_tr_use, feat_name, use_liq)
            X_te, _ = build_feature_matrix(df_test,  feat_name, use_liq)

            y_T_tr = df_tr_use['T_C'].values
            y_P_tr = df_tr_use['P_kbar'].values
            y_T_te = df_test['T_C'].values
            y_P_te = df_test['P_kbar'].values

            for model_name in ['RF', 'ERT', 'XGB', 'GB']:
                for target_name, y_tr, y_te_arr in [('T_C', y_T_tr, y_T_te),
                                                     ('P_kbar', y_P_tr, y_P_te)]:
                    pbar.set_postfix(trk=track_name, s=seed, f=feat_name,
                                     m=model_name, t=target_name)
                    frozen_key = (track_name, feat_name, model_name,
                                  target_name)
                    if frozen_key not in frozen_params_store:
                        logger.error('Missing frozen params: %s', frozen_key)
                        pbar.update(1)
                        continue
                    t1 = time.time()
                    try:
                        result = eval_frozen(
                            X_tr, y_tr, X_te, y_te_arr,
                            model_name, target_name,
                            frozen_params_store[frozen_key],
                        )
                        result.pop('model')
                        result['split_seed']  = seed
                        result['track']       = track_name
                        result['feature_set'] = feat_name
                        multi_seed_results.append(result)
                        logger.info(
                            '[FROZEN] %s s=%d %s %s %s rmse=%.2f r2=%.3f (%.0fs)',
                            track_name, seed, feat_name, model_name, target_name,
                            result['rmse_test'], result['r2_test'],
                            time.time() - t1,
                        )
                    except Exception as e:
                        logger.error('FAIL %s: %s: %s',
                                     frozen_key, type(e).__name__, e)
                    finally:
                        pbar.update(1)

        pd.DataFrame(multi_seed_results).to_csv(PARTIAL_LOCAL, index=False)
        elapsed_h = (time.time() - t0_global) / 3600
        logger.info('CHECKPOINT %s seed=%d | %d rows | %.2fh',
                    track_name, seed, len(multi_seed_results), elapsed_h)

pbar.close()

multi_seed_df = pd.DataFrame(multi_seed_results)
n_nan = multi_seed_df['rmse_test'].isna().sum()
if n_nan > 0:
    logger.error('CRITICAL: %d NaN rmse_test rows detected', n_nan)
    print(f'WARNING: {n_nan} NaN rows detected. Check log file.')

multi_seed_df['best_params'] = multi_seed_df['best_params'].astype(str)
multi_seed_df.to_csv(RESULTS / 'nb03_multi_seed_results.csv', index=False)
if PARTIAL_LOCAL.exists():
    PARTIAL_LOCAL.unlink()

total_h = (time.time() - t0_global) / 3600
logger.info('COMPLETE: %d rows in %.2fh', len(multi_seed_df), total_h)
print(f'\nFinal training complete: {len(multi_seed_df)} rows in {total_h:.2f}h')
print(f'NaN rows: {n_nan}')

## Phase 3.4b: heatmaps and statistical analysis

Three figures plus printed summary tables:
1. Seed-42 RMSE heatmap (single split, sanity check)
2. 10-seed mean RMSE heatmap with std as overlay
3. Multi-seed boxplot of test RMSE distributions per (model, feature)

In [ ]:
# ===== Multi-seed summary table =====
feat_order = ['raw', 'alr', 'pwlr']
model_order = ['RF', 'ERT', 'XGB', 'GB']
panels = [('opx_liq', 'T_C'), ('opx_liq', 'P_kbar'),
          ('opx_only', 'T_C'), ('opx_only', 'P_kbar')]

print('=' * 70)
print('MULTI-SEED RESULTS: mean +/- std test RMSE across 10 splits')
print('=' * 70)
for (track, target), grp in multi_seed_df.groupby(['track', 'target']):
    print(f'\n--- {track} / {target} ---')
    agg = (grp.groupby(['feature_set', 'model_name'])['rmse_test']
           .agg(['mean', 'std']))
    agg['display'] = [f'{m:.2f} +/- {s:.2f}' for m, s in
                      zip(agg['mean'], agg['std'])]
    pivot = agg['display'].unstack('model_name')
    pivot = pivot.reindex(feat_order)[model_order]
    print(pivot.to_string())

In [ ]:
# ===== Seed-42 RMSE heatmap =====
seed42 = multi_seed_df[multi_seed_df['split_seed'] == TUNE_SEED]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (track, target) in zip(axes.flat, panels):
    sub = seed42[(seed42['track'] == track) & (seed42['target'] == target)]
    pivot = sub.pivot(index='feature_set', columns='model_name',
                      values='rmse_test')
    pivot = pivot.reindex(feat_order)[model_order]
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='viridis_r',
                ax=ax, cbar_kws={'label': 'RMSE'},
                annot_kws={'size': 11})
    ax.set_title(f'{track} / {target}', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('Seed-42 benchmark: test RMSE (3 methods, no augmentation)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(FIGS / 'fig_nb03c_seed42_rmse.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ===== 10-seed mean RMSE heatmap with std overlay =====
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (track, target) in zip(axes.flat, panels):
    sub = multi_seed_df[(multi_seed_df['track'] == track) &
                        (multi_seed_df['target'] == target)]
    means = sub.groupby(['feature_set', 'model_name'])['rmse_test'].mean()
    stds = sub.groupby(['feature_set', 'model_name'])['rmse_test'].std()

    pivot_mean = means.unstack('model_name').reindex(feat_order)[model_order]
    pivot_std = stds.unstack('model_name').reindex(feat_order)[model_order]

    # build "mean\n+/- std" annotation matrix
    annot = pivot_mean.copy().astype(object)
    for i in range(pivot_mean.shape[0]):
        for j in range(pivot_mean.shape[1]):
            m = pivot_mean.iat[i, j]
            s = pivot_std.iat[i, j]
            annot.iat[i, j] = f'{m:.2f}\n±{s:.2f}'

    sns.heatmap(pivot_mean, annot=annot.values, fmt='', cmap='viridis_r',
                ax=ax, cbar_kws={'label': 'Mean RMSE'},
                annot_kws={'size': 9})
    ax.set_title(f'{track} / {target}', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.suptitle('10-seed mean test RMSE +/- std (3 methods, no augmentation)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(FIGS / 'fig_nb03c_multiseed_rmse.png', dpi=300,
            bbox_inches='tight')
plt.show()

In [ ]:
# ===== Boxplot: distribution of test RMSE across 10 seeds =====
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (track, target) in zip(axes.flat, panels):
    sub = multi_seed_df[(multi_seed_df['track'] == track) &
                        (multi_seed_df['target'] == target)].copy()
    sub['feature_set'] = pd.Categorical(sub['feature_set'],
                                         categories=feat_order, ordered=True)
    sub['model_name'] = pd.Categorical(sub['model_name'],
                                        categories=model_order, ordered=True)
    sns.boxplot(data=sub, x='feature_set', y='rmse_test', hue='model_name',
                ax=ax, palette='Set2')
    ax.set_title(f'{track} / {target}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature set')
    ax.set_ylabel('Test RMSE')
    ax.legend(title='Model', fontsize=8, loc='best')

fig.suptitle('Test RMSE distribution across 10 seeds',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(FIGS / 'fig_nb03c_multiseed_boxplot.png', dpi=300,
            bbox_inches='tight')
plt.show()

In [ ]:
# ===== Pairwise feature-set comparison: Wilcoxon signed-rank =====
# Tests whether log-ratio features (alr, pwlr) significantly differ from raw
# baseline within each (track, target, model) combo.
print('=' * 70)
print('FEATURE SET COMPARISON: Wilcoxon signed-rank vs raw baseline')
print('=' * 70)
for (track, target), grp in multi_seed_df.groupby(['track', 'target']):
    print(f'\n--- {track} / {target} ---')
    for mn in model_order:
        raw_arr = (grp[(grp['feature_set'] == 'raw') &
                       (grp['model_name'] == mn)]
                   .sort_values('split_seed')['rmse_test'].values)
        for alt in ['alr', 'pwlr']:
            alt_arr = (grp[(grp['feature_set'] == alt) &
                           (grp['model_name'] == mn)]
                       .sort_values('split_seed')['rmse_test'].values)
            if len(raw_arr) == len(alt_arr) and len(raw_arr) > 0:
                delta = raw_arr - alt_arr  # positive = alt is better
                try:
                    stat, p = stats.wilcoxon(delta, alternative='two-sided')
                except ValueError:
                    p = np.nan
                direction = 'BETTER' if delta.mean() > 0 else 'WORSE'
                sig = ('***' if p < 0.001 else '**' if p < 0.01
                       else '*' if p < 0.05 else '')
                print(f'  [{mn:4s}] raw vs {alt:5s}: '
                      f'delta={delta.mean():+.3f}  p={p:.4f} {sig:3s} '
                      f'{alt} is {direction}')

print('\n* = p<0.05, ** = p<0.01, *** = p<0.001')
logger.info('Diagnostic plots and statistical analysis complete.')

## Phase 3.5: winner selection and canonical model save

In [ ]:
# Phase 3.5: per-family winner selection (v7 Part B)
#
# Two families (forest = RF+ERT, boosted = GB+XGB). For each family and
# each (target, track), pick the (feature_set, model_name) with lowest
# 20-seed mean test RMSE. Tiebreaker: if another family candidate is
# within 1 std of the minimum, prefer the family's `tiebreaker_preferred`
# (RF for forest, XGB for boosted).

agg = multi_seed_df.groupby(
    ['track', 'target', 'feature_set', 'model_name']
).agg(
    rmse_test_mean=('rmse_test', 'mean'),
    rmse_test_std=('rmse_test', 'std'),
    r2_test_mean=('r2_test', 'mean'),
    r2_test_std=('r2_test', 'std'),
    overfit_ratio_mean=('overfit_ratio', 'mean'),
).reset_index()
agg.to_csv(RESULTS / 'nb03_multi_seed_summary.csv', index=False)


def _choose_family_winner(fam_subset, pref_model):
    """Pick min-RMSE row; if pref_model is within 1 std, prefer it."""
    if fam_subset.empty:
        return None
    min_row = fam_subset.loc[fam_subset['rmse_test_mean'].idxmin()]
    tol = float(min_row['rmse_test_std']) if pd.notna(min_row['rmse_test_std']) else 0.0
    band = fam_subset[
        fam_subset['rmse_test_mean'] <= min_row['rmse_test_mean'] + tol
    ]
    if pref_model in band['model_name'].values:
        pref_rows = band[band['model_name'] == pref_model]
        return pref_rows.loc[pref_rows['rmse_test_mean'].idxmin()]
    return min_row


per_family_winners = {
    'forest_family': {},
    'boosted_family': {},
    'tiebreaker_rule': TIEBREAKER_RULE,
    'selection_metadata': {
        'n_seeds': int(N_SPLIT_REPS),
        'split_seeds': list(SPLIT_SEEDS),
        'feature_methods': list(FEATURE_METHODS),
    },
}

canonical_files_written = []

for family_name, fam_cfg in MODEL_FAMILIES.items():
    fam_key = f'{family_name}_family'
    for track in ['opx_only', 'opx_liq']:
        for target in ['T_C', 'P_kbar']:
            subset = agg[
                (agg['track'] == track) &
                (agg['target'] == target) &
                (agg['model_name'].isin(fam_cfg['candidates']))
            ]
            chosen = _choose_family_winner(subset, fam_cfg['tiebreaker_preferred'])
            if chosen is None:
                print(f'WARNING: no candidates for {family_name}/{track}/{target}')
                continue
            model_name = str(chosen['model_name'])
            feat_set = str(chosen['feature_set'])
            filename = f'model_{target}_{track}_{family_name}.joblib'
            min_row = subset.loc[subset['rmse_test_mean'].idxmin()]
            winning_model = seed42_models.get(
                (track, target, model_name, feat_set)
            )
            if winning_model is None:
                print(f'WARNING: missing seed42 model for '
                      f'({track}, {target}, {model_name}, {feat_set})')
                continue
            joblib.dump(winning_model, MODELS / filename)
            canonical_files_written.append(filename)
            per_family_winners[fam_key][f'{track}_{target}'] = {
                'model_name': model_name,
                'feature_set': feat_set,
                'filename': filename,
                'rmse_test_mean': float(chosen['rmse_test_mean']),
                'rmse_test_std': float(chosen['rmse_test_std']),
                'r2_test_mean': float(chosen['r2_test_mean']),
                'r2_test_std': float(chosen['r2_test_std']),
                'tiebreaker_applied': bool(model_name != min_row['model_name']),
            }

with open(RESULTS / 'nb03_per_family_winners.json', 'w') as f:
    json.dump(per_family_winners, f, indent=2)

# Flat summary CSV for quick scanning / reporting.
flat_rows = []
for fam in ['forest_family', 'boosted_family']:
    for track in ['opx_only', 'opx_liq']:
        for target in ['T_C', 'P_kbar']:
            spec = per_family_winners[fam].get(f'{track}_{target}')
            if spec is None:
                continue
            flat_rows.append({
                'family': fam.replace('_family', ''),
                'track': track,
                'target': target,
                'model_name': spec['model_name'],
                'feature_set': spec['feature_set'],
                'rmse_test_mean': spec['rmse_test_mean'],
                'rmse_test_std': spec['rmse_test_std'],
                'r2_test_mean': spec['r2_test_mean'],
                'filename': spec['filename'],
                'tiebreaker_applied': spec['tiebreaker_applied'],
            })
pd.DataFrame(flat_rows).to_csv(
    RESULTS / 'nb03_per_family_winners_flat.csv', index=False
)

print(f'Per-family winners written: {RESULTS / "nb03_per_family_winners.json"}')
print(f'Canonical joblibs written: {len(canonical_files_written)}')
for fn in canonical_files_written:
    print(f'  {fn}')


## Phase 3.6: save canonical test prediction arrays

In [ ]:
# Phase 3.6: canonical test predictions per (family, target, track)
from src.data import canonical_model_path, load_per_family_winners

_winners = load_per_family_winners(RESULTS)
_test_idx_liq = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
_test_idx_opx = np.load(DATA_SPLITS / 'test_indices_opx.npy')
_df_test_liq = df_liq.loc[_test_idx_liq].copy()
_df_test_opx = df_opx.loc[_test_idx_opx].copy()

for _family in ['forest', 'boosted']:
    _fam_block = _winners[f'{_family}_family']
    for _track, _df_test_track, _use_liq in [
        ('opx_liq', _df_test_liq, True),
        ('opx_only', _df_test_opx, False),
    ]:
        _records = {
            'idx': _df_test_track.index,
            'y_T_true': _df_test_track['T_C'].values,
            'y_P_true': _df_test_track['P_kbar'].values,
        }
        for _target in ['T_C', 'P_kbar']:
            _spec = _fam_block.get(f'{_track}_{_target}')
            if _spec is None:
                continue
            _X_test, _ = build_feature_matrix(
                _df_test_track, _spec['feature_set'], _use_liq
            )
            _mdl = joblib.load(
                canonical_model_path(_target, _track, _family, MODELS, RESULTS)
            )
            _records[f'{_target}_pred'] = predict_median(_mdl, _X_test)
        pd.DataFrame(_records).to_csv(
            RESULTS / f'nb03_canonical_test_predictions_{_track}_{_family}.csv',
            index=False,
        )
print('Per-family canonical test predictions saved.')


## Phase 3.7: verification

Hardened against partial / failed runs.

In [ ]:
# Phase 3.7: verification (v7 Part B/D/G)
errors = []

required_files = [
    RESULTS / 'nb03_per_family_winners.json',
    RESULTS / 'nb03_per_family_winners_flat.csv',
    RESULTS / 'nb03_multi_seed_results.csv',
    RESULTS / 'nb03_multi_seed_summary.csv',
    DATA_SPLITS / 'train_indices_opx_liq.npy',
    DATA_SPLITS / 'test_indices_opx_liq.npy',
    DATA_SPLITS / 'train_indices_opx.npy',
    DATA_SPLITS / 'test_indices_opx.npy',
]
for _path in required_files:
    if not _path.exists():
        errors.append(f'MISSING: {_path}')

expected_rows = N_SPLIT_REPS * 2 * len(FEATURE_METHODS) * 4 * 2
actual_rows = len(multi_seed_df)
if actual_rows < expected_rows * 0.95:
    errors.append(
        f'multi_seed_df too sparse: got {actual_rows}, expected {expected_rows}'
    )

n_nan = multi_seed_df['rmse_test'].isna().sum()
if n_nan > 0:
    errors.append(f'multi_seed_df has {n_nan} NaN rmse_test rows')

with open(RESULTS / 'nb03_per_family_winners.json') as f:
    wjson = json.load(f)

for fam in ('forest_family', 'boosted_family'):
    fam_block = wjson.get(fam)
    if not fam_block:
        errors.append(f'per_family_winners missing {fam} block')
        continue
    for track in ('opx_only', 'opx_liq'):
        for target in ('T_C', 'P_kbar'):
            k = f'{track}_{target}'
            spec = fam_block.get(k)
            if spec is None:
                errors.append(f'missing winner: {fam}/{k}')
                continue
            fname = spec.get('filename')
            if not fname or not (MODELS / fname).exists():
                errors.append(f'missing canonical joblib: {MODELS / (fname or "")}')

if errors:
    print('=== VERIFICATION FAILED ===')
    for e in errors:
        print(f'  - {e}')
    raise AssertionError(f'{len(errors)} verification errors')

print('=== NB03 complete ===')
print(f'  {len(multi_seed_df)} multi-seed rows (expected {expected_rows})')
print(f'  two families (forest, boosted); 8 canonical joblibs written')
print(f'  selection source of truth: {RESULTS / "nb03_per_family_winners.json"}')


## Phase 3.8: Model-family ceiling analysis

**Purpose:** Test whether the RandomForest test RMSE is a property of the
winning feature set (ceiling) rather than a property of the RF family itself.
Evaluate six model families on the same canonical opx-liq split + feature set
used upstream in this notebook.

**Interpretation:** If the six families cluster within ~0.5 kbar / 20 C, the
model is feature-limited and performance is at or near the information ceiling
for the current input set. This motivates the H2O-engineered-feature work
elsewhere in the paper.

**Outputs:** `results/nb11_model_family_ceiling.csv`,
`figures/fig_nb11_model_family_ceiling.{png,pdf}` (names retained so previous
artifact references elsewhere still resolve).

In [28]:
# Phase 3.8 setup: load canonical split + RF params from NB03 multi-seed table.
import ast
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt

from src.features import build_feature_matrix

# v7 hotfix: per-family canonical resolves the ceiling feature set.
from src.data import canonical_model_spec
_spec_ceil = canonical_model_spec('T_C', 'opx_liq', 'forest', RESULTS)
WIN_FEAT_CEIL = _spec_ceil['feature_set']
print(f'Ceiling analysis feature set (forest/T_C/opx_liq): {WIN_FEAT_CEIL}')

df_liq_c = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
idx_tr = np.load(DATA_SPLITS / 'train_indices_opx_liq.npy')
idx_te = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')
df_tr_c = df_liq_c.loc[idx_tr].copy()
df_te_c = df_liq_c.loc[idx_te].copy()
X_tr_c, _ = build_feature_matrix(df_tr_c, WIN_FEAT_CEIL, use_liq=True)
X_te_c, _ = build_feature_matrix(df_te_c, WIN_FEAT_CEIL, use_liq=True)
yT_tr, yP_tr = df_tr_c['T_C'].values, df_tr_c['P_kbar'].values
yT_te, yP_te = df_te_c['T_C'].values, df_te_c['P_kbar'].values

def _parse_params(s):
    if isinstance(s, dict): return s
    try: return ast.literal_eval(s)
    except Exception:
        try: return json.loads(s)
        except Exception: return {}

multi_c = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
rf_liq_c = multi_c[(multi_c.track == 'opx_liq') & (multi_c.model_name == 'RF') &
                   (multi_c.feature_set == WIN_FEAT_CEIL) & (multi_c.split_seed == 42)]
rf_params_T = _parse_params(rf_liq_c[rf_liq_c.target == 'T_C'].iloc[0]['best_params'])
rf_params_P = _parse_params(rf_liq_c[rf_liq_c.target == 'P_kbar'].iloc[0]['best_params'])
print(f'train n={len(df_tr_c)}  test n={len(df_te_c)}  n_features={X_tr_c.shape[1]}')
print(f'RF T params: {rf_params_T}')
print(f'RF P params: {rf_params_P}')


Ceiling analysis feature set (forest/T_C/opx_liq): alr
train n=426  test n=174  n_features=23
RF T params: {'n_estimators': 800, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.66, 'max_depth': 20}
RF P params: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.66, 'max_depth': 10}


In [29]:
# Phase 3.8: build six model-family configurations.
SEED_CEIL = 42

def _mk_rf(params):
    return RandomForestRegressor(**params, random_state=SEED_CEIL, n_jobs=-1)
def _mk_et():
    return ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2,
                               max_features='sqrt',
                               random_state=SEED_CEIL, n_jobs=-1)
def _mk_xgb():
    return xgb.XGBRegressor(n_estimators=800, max_depth=6, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            reg_alpha=0.1, reg_lambda=1.0,
                            tree_method='hist', random_state=SEED_CEIL,
                            n_jobs=-1, verbosity=0)
def _mk_mlp(hidden):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MLPRegressor(hidden_layer_sizes=hidden, activation='relu',
                             solver='adam', alpha=1e-3, learning_rate_init=1e-3,
                             max_iter=800, early_stopping=True,
                             validation_fraction=0.1, random_state=SEED_CEIL)),
    ])
def _mk_ridge():
    return Pipeline([('scaler', StandardScaler()),
                     ('ridge', Ridge(alpha=1.0, random_state=SEED_CEIL))])

FAMILIES_CEIL = [
    ('RF_baseline',    lambda: _mk_rf(rf_params_T), lambda: _mk_rf(rf_params_P)),
    ('ExtraTrees',     _mk_et,     _mk_et),
    ('XGBoost_tuned',  _mk_xgb,    _mk_xgb),
    ('MLP_32_64_32',   lambda: _mk_mlp((32, 64, 32)),  lambda: _mk_mlp((32, 64, 32))),
    ('MLP_64_128_64',  lambda: _mk_mlp((64, 128, 64)), lambda: _mk_mlp((64, 128, 64))),
    ('Ridge',          _mk_ridge,  _mk_ridge),
]
print(f'Testing {len(FAMILIES_CEIL)} model families.')


Testing 6 model families.


In [30]:
# Phase 3.8: fit each family and save metrics.
def _eval_one(name, factory, X_tr, y_tr, X_te, y_te):
    model = factory()
    model.fit(X_tr, y_tr)
    return {
        'family': name,
        'rmse_train': float(np.sqrt(mean_squared_error(y_tr, model.predict(X_tr)))),
        'rmse_test':  float(np.sqrt(mean_squared_error(y_te, model.predict(X_te)))),
        'mae_test':   float(mean_absolute_error(y_te, model.predict(X_te))),
        'r2_test':    float(r2_score(y_te, model.predict(X_te))),
    }

rows_ceil = []
for name, fac_T, fac_P in FAMILIES_CEIL:
    print(f'  fitting {name} ...', flush=True)
    r_T = _eval_one(name, fac_T, X_tr_c, yT_tr, X_te_c, yT_te); r_T['target'] = 'T_C'
    r_P = _eval_one(name, fac_P, X_tr_c, yP_tr, X_te_c, yP_te); r_P['target'] = 'P_kbar'
    rows_ceil += [r_T, r_P]

df_fam = pd.DataFrame(rows_ceil)[['target', 'family', 'rmse_train',
                                  'rmse_test', 'mae_test', 'r2_test']]
df_fam.to_csv(RESULTS / 'nb11_model_family_ceiling.csv', index=False)

print('\n== Model-family test RMSE ==')
for tgt in ['T_C', 'P_kbar']:
    sub = df_fam[df_fam.target == tgt].sort_values('rmse_test')
    print(f'\n{tgt}:')
    print(sub[['family', 'rmse_test', 'mae_test', 'r2_test']].round(3).to_string(index=False))
    rng = sub['rmse_test'].max() - sub['rmse_test'].min()
    unit = 'C' if tgt == 'T_C' else 'kbar'
    print(f'  family RMSE spread: {rng:.2f} {unit}')


  fitting RF_baseline ...
  fitting ExtraTrees ...
  fitting XGBoost_tuned ...
  fitting MLP_32_64_32 ...
  fitting MLP_64_128_64 ...
  fitting Ridge ...

== Model-family test RMSE ==

T_C:
       family  rmse_test  mae_test  r2_test
        Ridge     86.835    59.184    0.608
  RF_baseline     87.545    60.014    0.602
XGBoost_tuned     93.608    65.374    0.545
MLP_64_128_64    100.584    76.594    0.474
   ExtraTrees    105.957    67.904    0.417
 MLP_32_64_32    151.612   119.383   -0.195
  family RMSE spread: 64.78 C

P_kbar:
       family  rmse_test  mae_test  r2_test
 MLP_32_64_32      5.057     3.874    0.725
  RF_baseline      5.316     3.785    0.696
XGBoost_tuned      5.435     3.903    0.682
MLP_64_128_64      6.263     4.596    0.578
        Ridge      6.639     5.107    0.526
   ExtraTrees      7.644     5.271    0.371
  family RMSE spread: 2.59 kbar


In [31]:
# Phase 3.8: horizontal bar figure.
COLORS_CEIL = {
    'RF_baseline': '#0072B2', 'ExtraTrees': '#56B4E9',
    'XGBoost_tuned': '#D55E00', 'MLP_32_64_32': '#009E73',
    'MLP_64_128_64': '#117755', 'Ridge': '#CC79A7',
}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
for ax, tgt, units in zip(axes, ['T_C', 'P_kbar'], ['C', 'kbar']):
    sub = df_fam[df_fam.target == tgt].sort_values('rmse_test')
    colors = [COLORS_CEIL.get(f, '#777777') for f in sub['family']]
    bars = ax.barh(sub['family'], sub['rmse_test'], color=colors, edgecolor='black')
    rf_rmse = float(sub.loc[sub.family == 'RF_baseline', 'rmse_test'].iloc[0])
    ax.axvline(rf_rmse, color='#0072B2', linestyle='--', linewidth=1.2,
               alpha=0.6, label=f'RF baseline = {rf_rmse:.2f} {units}')
    for b, v in zip(bars, sub['rmse_test']):
        ax.text(v + 0.01*v, b.get_y() + b.get_height()/2,
                f'{v:.2f}', va='center', fontsize=9)
    ax.set_xlabel(f'Test RMSE ({units})')
    ax.set_title(f'{tgt}: model-family ceiling (n_test={len(df_te_c)})')
    ax.legend(loc='lower right', fontsize=8, framealpha=0.9)
    ax.grid(axis='x', alpha=0.3)
fig.suptitle('Model-family ceiling: six families on canonical feature set',
             fontsize=11)
fig.savefig(FIGURES / 'fig_nb11_model_family_ceiling.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURES / 'fig_nb11_model_family_ceiling.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_nb11_model_family_ceiling.{png,pdf}')

# Verification
assert (RESULTS / 'nb11_model_family_ceiling.csv').exists()
chk = pd.read_csv(RESULTS / 'nb11_model_family_ceiling.csv')
assert set(chk['target']) == {'T_C', 'P_kbar'}
expected = {'RF_baseline', 'ExtraTrees', 'XGBoost_tuned',
            'MLP_32_64_32', 'MLP_64_128_64', 'Ridge'}
assert set(chk['family']) == expected


Saved fig_nb11_model_family_ceiling.{png,pdf}


**Takeaway:** The RF baseline sits within the model-family cluster rather
than leading it, which supports the interpretation that the remaining test
RMSE is a feature-set (information) ceiling rather than a failure of the RF
family. Adding richer features (H2O-engineered channels in later work) is the
most likely path to move the ceiling.